# B-4 后训练（BP RoboTwin split v1）checkpoint 评估与报告图

两段：**B-4 SFT**（B-4 10k 起，LoRA + 示教编码器，20k 步）→ **B-4 SFT full**（B-4 SFT 20k 合并 LoRA 后起，Qwen 语言模型全量 + Action Expert + 示教编码器，ViT 冻结，30k 步）。跨阶段图的横轴为累计步数（B-4 SFT full 的第 0 步 = B-4 SFT 的 20k）。

代码与图沿用 `stage_b_ckpt_eval.ipynb`（同样的图号与风格），输入改为本机 B-4 后训练 run 与 BP RoboTwin 的 G9；增加 BP 专有的 `goal` 条件（同锚点、另一目标的示教）与按分组 / 任务 / 时刻的图 12-15，最后附逐任务与仿真测评对照。

只读取 G9 JSON 和训练 `loss.log`，不加载模型、不用 GPU、不影响训练。

## 使用方法

1. 在下一格的 `STAGES` 里列出要看的阶段（默认 B-3、B-4）：`run_root` 为阶段输出目录（只有一个 run 时可填父目录，否则填具体时间戳 run），`eval_glob` 为该阶段的 G9 文件，`start_step` 为该阶段第 0 步在累计步数轴上的位置（B-4 从 B-3 105k 初始化 → 105000，只用于对比图）。
2. 按顺序运行全部单元格；新增 checkpoint 或 G9 后从“读取”单元格开始重跑。
3. 每个阶段各自一套图；只有训练日志、还没有 G9 的阶段（如刚启动的 B-4）只画训练曲线；还没有 `loss.log` 的阶段跳过。至少两个阶段有数据时生成跨阶段对比图。
4. 最后一格导出 300 DPI PNG + 矢量 PDF/SVG + CSV + `manifest.json`：各阶段到 `<run>/reports/checkpoint_eval/`，对比图到 `stage_b/reports/stage_compare/`。同名文件会覆盖，需要固定某次报告时改 `EXPORT_DIRS` / `COMPARE_DIR`。

## 指标口径

- `train_loss_100 / 1000 / 5000`：截至 checkpoint 的最近 100 / 1,000 / 5,000 步 flow-matching 训练 loss 均值；窗口不完整为 NaN。`loss.log` 是主进程记录，**不是 8 卡全局均值**。
- `matched / blank / alt / wrong`：采样动作在归一化 delta 动作空间的 MSE（越低越好），**不是 EE 厘米误差**，也不能和训练 flow loss 比绝对值。
- 比值与 95% CI 直接读 G9 的配对 query-grouped bootstrap；`alt` 只覆盖可替换的 query，分母是同一子集的 matched。CI 是各 checkpoint 内的条件比值 CI，不是跨 checkpoint 差异的 CI。
- **B-4 的 G9** 以 P5 特征替换示教、由 B-4 自己的编码器在线编码；与 B-3（替换缓存 `Z_demo`）可比：初始 B-4 与 B-3 105k 的 G9 差异 ≤ 0.3%。
- 不同 query 数的评估分开连线；旧 JSON 没有 query ID / seed，相同数量不证明是同一批 query。不自动判定收敛、显著性或最佳 checkpoint。
- 图中文字用英文（无中文字体环境也能导出）；跨臂跟随专门测评按此前决定暂缓。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import io
import json
import math

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

REPO_ROOT = Path('/vla/workspace/my_tbot')
STAGE_B = REPO_ROOT / 'outputs/BPv4CDX/stage_b'
G9_DIR = STAGE_B / 'g9_bp'

# ===== 主要输入：每个阶段一条 =====
# start_step：该阶段第 0 步在累计步数轴上的位置（只用于对比图）。B-4 从 B-3 105k 初始化。
# eval_glob 可以是多个 glob：两套 G9 查询数不同（g9_main 600 / g9_goal 444），按 query 数分开连线
#   results/step*.json          tools/bpv4cdxb/g9_bprobotwin.py（gate_g9_main）：留出候选集 / 留出任务 / 训练兄弟集
#   2026-10-29_18-26_*k.json    gate_g9_bp：训练 38 任务 / 留出候选集 / 留出任务，含 goal（同锚点另一目标）
#   2026-10-30_02-46_*k.json    B-4 SFT full 的 gate_g9_bp（与 B-4 SFT 的 g9_goal 同一份 444 查询索引，可直接比较）
STAGES = [
    {'name': 'B-4 SFT', 'run_root': STAGE_B / 'b4_sft/2026-10-29_18-26/2026-10-29_18-27',
     'eval_glob': ('results/step*.json', '2026-10-29_18-26_*k.json'), 'start_step': 0},
    {'name': 'B-4 SFT full', 'run_root': STAGE_B / 'b4_sft_full/2026-10-30_02-46/2026-10-30_02-47',
     'eval_glob': '2026-10-30_02-46_*k.json', 'start_step': 20000},
]

# 同一阶段同一 checkpoint、同一 query 数有重复评估时报错，用这里排除旧文件（例如 {'b2_010k.json'}）
EXCLUDE_FILES = set()
QUERY_COUNTS = None      # 例如 [100]；None = 全部，按 query 数分开连线

MIN_STEP = 0             # 各阶段自己的步数范围
MAX_STEP = None
ZOOM_FROM_STEP = {'B-4 SFT': 2000, 'B-4 SFT full': 2000}  # 训练 loss 放大图起点；未列出的阶段 = 0
LOG_FREQ_OVERRIDE = None # None = 从 checkpoint 的 train_config.json 读，还没有 checkpoint 时由 loss.log 步长推断
SMOOTH_STEPS = 1000      # 训练曲线只做 trailing mean
WINDOW_STEPS = (100, 1000, 5000)
SELECT_STEP = {}         # 分组 CI 图的 checkpoint，如 {'B-3': 65000}；未列出 = 该阶段最新已评估
SELECT_QUERIES = 444     # 分组图用 g9_goal（444 queries，含 goal / 分组 / 任务）；None = 最大的 query 数

# ===== 报告输出 =====
EXPORT = True
EXPORT_DIRS = {}         # 各阶段导出目录；未列出 = <run>/reports/checkpoint_eval
COMPARE_DIR = STAGE_B / 'reports' / 'stage_compare'
FORMATS = ('png', 'pdf', 'svg')
DPI = 300
FIGSIZE_ROW = (14, 4.3)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.22,
    'legend.fontsize': 8, 'figure.dpi': 110, 'figure.max_open_warning': 0,  # 图留到最后统一导出
    'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'svg.fonttype': 'none',
})
pd.set_option('display.max_columns', 24)
pd.set_option('display.width', 200)
COLORS = {'matched': '#0072B2', 'blank': '#D55E00', 'alt': '#009E73', 'wrong': '#CC79A7', 'goal': '#E69F00'}
STAGE_COLORS = {'B-2': '#7F7F7F', 'B-3': '#0072B2', 'B-4': '#D55E00', 'B-4 SFT': '#0072B2', 'B-4 SFT full': '#D55E00'}
MARKERS = ('o', 's', '^', 'D')
LINESTYLES = ('-', '--', ':', '-.')

In [ ]:
# ===== 读取：每个阶段一个 ctx（新增 checkpoint / G9 后从这里重跑） =====
def clipped(df):
    out = df[df['step'] >= MIN_STEP]
    if MAX_STEP is not None:
        out = out[out['step'] <= MAX_STEP]
    return out.sort_values('step').reset_index(drop=True)


def numeric(value):
    try:
        return float(value) if value is not None else np.nan
    except (TypeError, ValueError):
        return np.nan


def resolve_run(root):
    root = Path(root).expanduser().resolve()
    if (root / 'loss.log').is_file():
        return root
    candidates = sorted(p.parent for p in root.glob('*/loss.log'))
    if len(candidates) > 1:
        raise ValueError(f'{root} 下有 {len(candidates)} 个 run，请把 run_root 改成具体时间戳目录：{candidates}')
    return candidates[0].resolve() if candidates else None


def load_loss(path, issues):
    # 日志仍在写入时只取到最后一个换行为止的完整行
    text = path.read_text(encoding='utf-8')
    text = text[:text.rfind('\n') + 1]
    rows = []
    for r in csv.DictReader(io.StringIO(text)):
        try:
            step, loss = int(r['step']), float(r['loss'])
            if step < 1 or not np.isfinite(loss):
                raise ValueError('invalid step/loss')
        except (TypeError, KeyError, ValueError):
            issues.append(f'{path}: skipped malformed CSV row')
            continue
        rows.append({**{k: numeric(v) for k, v in r.items() if k != 'timestamp'},
                     'step': step, 'loss': loss, 'timestamp': r.get('timestamp')})
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.drop_duplicates('step', keep='last').sort_values('step').reset_index(drop=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
    df['elapsed_hours'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds() / 3600
    return df


def train_config(run):
    files = sorted((run / 'checkpoints').glob('[0-9]*/pretrained_model/train_config.json'))
    return (json.loads(files[-1].read_text()), files[-1]) if files else ({}, None)


def log_freq(train, cfg_json):
    if LOG_FREQ_OVERRIDE is not None:
        return int(LOG_FREQ_OVERRIDE)
    if 'log_freq' in cfg_json:
        return int(cfg_json['log_freq'])
    diffs = np.diff(train['step'].to_numpy())
    return int(np.median(diffs)) if len(diffs) else int(train['step'].iloc[0])


def read_evaluations(root, run, eval_glob, issues):
    records = []
    patterns = (eval_glob,) if isinstance(eval_glob, str) else tuple(eval_glob)
    for p in sorted({q for pattern in patterns for q in root.glob(pattern)}):
        if p.name in EXCLUDE_FILES:
            continue
        try:
            d = json.loads(p.read_text())
        except (OSError, json.JSONDecodeError) as exc:
            issues.append(f'{p.name}: skipped incomplete JSON: {exc}')
            continue
        try:
            ckpt = Path(d['ckpt']).expanduser()  # 权重文件不必仍存在，只用路径匹配 run
            ckpt = (ckpt if ckpt.is_absolute() else REPO_ROOT / ckpt).resolve()
            step, checkpoint_run = int(ckpt.parent.name), ckpt.parents[2]
        except (KeyError, ValueError, IndexError, TypeError):
            issues.append(f'{p.name}: invalid ckpt path')
            continue
        if checkpoint_run != run:
            continue
        queries = int(d.get('queries', 0))
        if QUERY_COUNTS is not None and queries not in QUERY_COUNTS:
            continue
        if step < MIN_STEP or (MAX_STEP is not None and step > MAX_STEP):
            continue
        if queries < 1 or not np.isfinite(numeric(d.get('mean_error', {}).get('matched'))):
            issues.append(f'{p.name}: invalid queries/matched')
            continue
        records.append({'step': step, 'queries': queries, 'path': p, 'data': d})
    seen = {}
    for rec in records:
        key = (rec['step'], rec['queries'])
        if key in seen:
            raise ValueError(f'重复评估 {key}：{seen[key]} / {rec["path"].name}；用 EXCLUDE_FILES 明确选择。')
        seen[key] = rec['path'].name
    return sorted(records, key=lambda r: (r['step'], r['queries']))


def ratio_fields(entry):
    entry = entry or {}
    ci = entry.get('ci95') or [np.nan, np.nan]
    if len(ci) != 2:
        ci = [np.nan, np.nan]
    return {'ratio': numeric(entry.get('ratio')), 'ci_low': numeric(ci[0]), 'ci_high': numeric(ci[1]),
            'n': numeric(entry.get('n')), 'groups': numeric(entry.get('groups'))}


def flatten_records(records):
    main, groups = [], []
    for rec in records:
        d = rec['data']
        row = {'step': rec['step'], 'queries': rec['queries'], 'samples': d.get('samples'),
               'eval_file': str(rec['path']), 'ckpt': d['ckpt'],
               'code_revision': d.get('code_revision'), 'G9_blank_gain': d.get('G9_blank_gain')}
        for metric, value in d.get('mean_error', {}).items():
            row[metric] = numeric(value)
        for condition, entry in d.get('ratios', {}).items():
            row.update({f'{condition}_{k}': v for k, v in ratio_fields(entry).items()})
        main.append(row)
        for dimension, field in [('robot', 'by_robot_blank'), ('source', 'by_source'), ('pool', 'alt_by_pool'),
                                 ('group', 'by_group'), ('task', 'by_task'), ('phase', 'by_phase')]:
            for name, detail in (d.get(field) or {}).items():
                entries = {'blank': detail} if dimension == 'robot' else {'alt': detail} if dimension == 'pool' else {
                    c: detail[c] for c in ('blank', 'alt', 'wrong', 'goal') if c in detail}
                if dimension in ('group', 'task', 'phase') and 'matched_mse' in detail:
                    groups.append({'step': rec['step'], 'queries': rec['queries'], 'dimension': dimension, 'group': name,
                                   'condition': 'matched', 'ratio': np.nan, 'ci_low': np.nan, 'ci_high': np.nan,
                                   'n': numeric(detail.get('samples')), 'groups': numeric(detail.get('queries')),
                                   'mse': numeric(detail['matched_mse'])})
                for condition, entry in entries.items():
                    groups.append({'step': rec['step'], 'queries': rec['queries'], 'dimension': dimension,
                                   'group': name, 'condition': condition, **ratio_fields(entry),
                                   'mse': numeric(detail.get(f'{condition}_mse')) if isinstance(detail, dict) else np.nan})
    return pd.DataFrame(main).sort_values(['step', 'queries']).reset_index(drop=True), pd.DataFrame(
        groups, columns=['step', 'queries', 'dimension', 'group', 'condition', 'ratio', 'ci_low', 'ci_high', 'n', 'groups', 'mse'])


def window_loss(losses, end, width, freq):
    # 每行是截至该 step 的 freq 步均值；窗口内的记录必须齐全
    if width % freq:
        return np.nan
    expected = list(range(end - width + freq, end + 1, freq))
    if not expected or expected[0] <= 0 or any(s not in losses for s in expected):
        return np.nan
    return float(np.mean([losses[s] for s in expected]))


def align_train(g9, train, freq):
    losses = train.set_index('step')['loss'].to_dict()
    out = g9.copy()
    for width in WINDOW_STEPS:
        out[f'train_loss_{width}'] = [window_loss(losses, int(s), width, freq) for s in out['step']]
    fields = [c for c in ('step', 'timestamp', 'elapsed_hours', 'lr', 'grad_norm', 'update_s', 'dataloading_s') if c in train]
    return out.merge(train[fields], on='step', how='left', validate='many_to_one')


def load_stage(spec):
    name, issues = spec['name'], []
    run = resolve_run(spec['run_root'])
    if run is None:
        print(f'{name}: 跳过——{spec["run_root"]} 下还没有 loss.log')
        return None
    train_all = load_loss(run / 'loss.log', issues)
    if train_all.empty:
        print(f'{name}: 跳过——{run}/loss.log 还没有完整记录')
        return None
    cfg_json, cfg_file = train_config(run)
    freq = log_freq(train_all, cfg_json)
    if SMOOTH_STEPS <= 0 or SMOOTH_STEPS % freq:
        raise ValueError(f'{name}: SMOOTH_STEPS 必须是 log_freq={freq} 的整数倍')
    records = read_evaluations(G9_DIR.expanduser().resolve(), run, spec['eval_glob'], issues)
    ctx = {'name': name, 'run': run, 'title': f'{name} ({run.name})', 'start': int(spec.get('start_step', 0)),
           'freq': freq, 'train_all': train_all, 'train': clipped(train_all), 'records': records,
           'ckpt': None, 'groups': None, 'select': None, 'cfg_file': cfg_file, 'issues': issues,
           'figures': {}, 'notes': [],
           'report_dir': Path(EXPORT_DIRS[name]).expanduser() if name in EXPORT_DIRS else run / 'reports' / 'checkpoint_eval'}
    if records:
        g9, ctx['groups'] = flatten_records(records)
        ctx['ckpt'] = align_train(g9, train_all, freq)
        step = int(SELECT_STEP.get(name, ctx['ckpt']['step'].max()))
        rows = ctx['ckpt'][ctx['ckpt']['step'] == step]
        if rows.empty:
            raise ValueError(f'{name}: SELECT_STEP={step} 没有评估')
        queries = int(SELECT_QUERIES if SELECT_QUERIES is not None else rows['queries'].max())
        if not (rows['queries'] == queries).any():
            raise ValueError(f'{name}: SELECT_QUERIES={queries} 在 {step} 步没有评估')
        ctx['select'] = (step, queries)
    return ctx


CTX = {}
for spec in STAGES:
    ctx = load_stage(spec)
    if ctx is not None:
        CTX[spec['name']] = ctx

COLUMNS = ['step', 'queries', 'train_loss_100', 'train_loss_1000', 'train_loss_5000',
           'matched', 'blank', 'alt', 'wrong', 'goal', 'blank_ratio', 'alt_ratio', 'wrong_ratio', 'goal_ratio']
for name, ctx in CTX.items():
    last = int(ctx['train']['step'].max()) if len(ctx['train']) else 0
    print(f"\n== {ctx['title']} ==\nrun: {ctx['run']}\n训练日志 {len(ctx['train_all'])} 行（到第 {last:,} 步）；log_freq={ctx['freq']}")
    if ctx['ckpt'] is None:
        print('尚无 G9 结果：只画训练曲线。')
    else:
        print(f"G9 {len(ctx['ckpt'])} 个；query 数 {sorted(ctx['ckpt'].queries.unique())}；分组图用 checkpoint={ctx['select'][0]:,}（{ctx['select'][1]} queries）")
        display(ctx['ckpt'][[c for c in COLUMNS if c in ctx['ckpt']]].round(6))
    if ctx['issues']:
        print('读取警告：\n' + '\n'.join(ctx['issues']))
print('\n注意：train_loss 是主进程 flow loss；matched 等是留出动作 MSE；两者量纲不同。')

In [ ]:
# ===== 绘图工具 =====
def finish_axis(ax, title, ylabel, xlabel='Training step (k)'):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x / 1000:g}'))
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False)


def register(target, name, fig, note):
    fig.suptitle(target['title'], fontsize=12)
    fig.text(0.01, 0.01, note, fontsize=7, color='0.35', va='bottom')
    fig.tight_layout(rect=(0, 0.07, 1, 0.96))
    target['figures'][name] = fig
    target['notes'].append({'figure': name, 'caption': note})
    plt.show()


def eval_caption(ctx):
    ck = ctx['ckpt']
    counts = ', '.join(str(int(q)) for q in sorted(ck['queries'].unique()))
    return (f"Source: G9 JSON; {ctx['name']} run={ctx['run'].name}; steps {int(ck.step.min()):,}-{int(ck.step.max()):,}; "
            f'queries={counts}. Same count does not verify identical query IDs/seeds.')


def train_smooth(df, metric, freq, width=SMOOTH_STEPS):
    # 缺日志处断开，不把不完整窗口当完整窗口
    k = width // freq
    series = df.set_index('step')[metric]
    steps = pd.Series(series.index, index=series.index)
    complete = steps - steps.shift(k - 1) == width - freq
    return series.rolling(k, min_periods=k).mean().where(complete).to_numpy()


def training_series(ax, df, metric, title, ylabel, freq, log=False, color=COLORS['matched'], offset=0, label=''):
    values = df[metric].to_numpy(dtype=float)
    x = df['step'] + offset
    ax.plot(x, values, color=color, lw=0.7, alpha=0.23, label=f'{label}logged {freq}-step mean (rank 0)')
    ax.plot(x, train_smooth(df, metric, freq), color=color, lw=1.8, label=f'{label}trailing {SMOOTH_STEPS:,}-step mean')
    if log and (values[np.isfinite(values)] > 0).all():
        ax.set_yscale('log')


def evaluation_series(ax, df, metric, color=None, offset=0, label=''):
    # 不同 query 数分开连线
    for i, (queries, group) in enumerate(df.groupby('queries', sort=True)):
        group = group.sort_values('step')
        ax.plot(group['step'] + offset, group[metric], marker=MARKERS[i % 4], linestyle=LINESTYLES[i % 4],
                color=color or COLORS.get(metric, COLORS['matched']), lw=1.6, ms=4,
                label=f'{label}{metric}; {int(queries)} queries')


def ratio_series(ax, df, condition, color=None, offset=0, label='', ci_label=True):
    key, low, high = f'{condition}_ratio', f'{condition}_ci_low', f'{condition}_ci_high'
    if key not in df:
        return
    color = color or COLORS[condition]
    for i, (queries, group) in enumerate(df.groupby('queries', sort=True)):
        group = group.sort_values('step')
        x = (group['step'] + offset).to_numpy()
        ax.plot(x, group[key], marker=MARKERS[i % 4], linestyle=LINESTYLES[i % 4], lw=1.6, ms=4, color=color,
                label=f'{label}{condition}; {int(queries)} queries')
        if low in group and high in group:
            ax.fill_between(x, group[low].to_numpy(dtype=float), group[high].to_numpy(dtype=float), color=color,
                            alpha=0.12, label='within-checkpoint paired bootstrap 95% CI' if ci_label and i == 0 else None)


def ratio_refs(ax, condition):
    ax.axhline(1.0, color='0.4', ls='--', lw=0.9, label='no error change (1.0)')
    if condition == 'blank':
        ax.axhline(1.1, color='0.65', ls=':', lw=0.8, label='G9 ratio threshold (1.1; CI also required)')


RATIO_YLABEL = 'Condition MSE / paired matched MSE (unitless)'
MSE_YLABEL = 'Normalized delta action MSE (unitless)'


def group_plot(ctx, dimension, title, name, allowed):
    step, queries = ctx['select']
    g = ctx['groups']
    subset = g[(g.step == step) & (g.queries == queries) & (g.dimension == dimension)]
    if subset.empty:
        return
    conditions = [c for c in allowed if (subset.condition == c).any()]
    labels = sorted(subset['group'].unique())
    if dimension == 'pool':
        order = ('2-3', '4-6', '7-15', '16+')
        labels = [x for x in order if x in labels] + [x for x in labels if x not in order]
    fig, axs = plt.subplots(1, len(conditions), figsize=(max(8, 5.5 * len(conditions)), max(3.5, 0.48 * len(labels) + 1.5)),
                            squeeze=False)
    for ax, condition in zip(axs[0], conditions):
        rows = subset[subset.condition == condition].set_index('group').reindex(labels)
        y = np.arange(len(labels))
        for yy, lo, hi in zip(y, rows.ci_low.to_numpy(dtype=float), rows.ci_high.to_numpy(dtype=float)):
            if np.isfinite(lo) and np.isfinite(hi):
                ax.plot([lo, hi], [yy, yy], color=COLORS[condition], lw=1.4)
        ax.scatter(rows.ratio, y, color=COLORS[condition], s=28, zorder=3, label=condition)
        ax.set_yticks(y, [f'{lab} (Q={int(n) if np.isfinite(n) else "?"})' for lab, n in zip(labels, rows['groups'])])
        ax.invert_yaxis()
        ax.axvline(1, color='0.45', ls='--', lw=0.9)
        ax.set_title(f'{condition.capitalize()} / paired matched: checkpoint {step:,}, {queries} queries')
        ax.set_xlabel('Error ratio with query-bootstrap 95% CI (unitless)')
        ax.set_ylabel(dimension.capitalize())
        ax.legend(frameon=False)
    register(ctx, name, fig, f"{title}. Source: {ctx['name']} G9 JSON at step {step:,}. Q = eligible query groups; "
             'tiny-Q intervals may be degenerate and are not reliable evidence of precision.')


# ===== 各阶段报告图 =====
def stage_report(ctx):
    train, freq, ck = ctx['train'], ctx['freq'], ctx['ckpt']
    tcap = f"Source: {ctx['name']} {ctx['run'].name}/loss.log; rank-0 logged {freq}-step means; smoothing={SMOOTH_STEPS:,} trailing steps."
    if ck is not None:
        # 01 总览：训练 loss、matched、blank / wrong 比值（量纲分开）
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        training_series(axes[0, 0], train, 'loss', 'Training flow-matching loss', 'Flow loss (unitless)', freq)
        finish_axis(axes[0, 0], 'Training flow-matching loss', 'Flow loss (unitless)')
        evaluation_series(axes[0, 1], ck, 'matched')
        finish_axis(axes[0, 1], 'Held-out matched action error', MSE_YLABEL)
        for ax, cond, t in ((axes[1, 0], 'blank', 'Blank-demo error ratio'), (axes[1, 1], 'wrong', 'Wrong-task-demo error ratio')):
            ratio_series(ax, ck, cond)
            ratio_refs(ax, cond)
            finish_axis(ax, t, RATIO_YLABEL)
        register(ctx, '01_overview', fig, eval_caption(ctx) + ' Training loss: rank-0 interval means; shaded bands are ratio CI only.')

    # 02 训练 loss 全程 + 后期放大
    zoom = train[train.step >= ZOOM_FROM_STEP.get(ctx['name'], 0)]
    panels = 2 if len(zoom) and len(zoom) < len(train) else 1
    fig, axs = plt.subplots(1, panels, figsize=FIGSIZE_ROW if panels == 2 else (8, 4.3), squeeze=False)
    training_series(axs[0, 0], train, 'loss', '', '', freq, log=True)
    finish_axis(axs[0, 0], 'Training flow loss: full range (log y)', 'Flow loss (unitless)')
    if panels == 2:
        training_series(axs[0, 1], zoom, 'loss', '', '', freq)
        finish_axis(axs[0, 1], f"Training flow loss: step >= {ZOOM_FROM_STEP.get(ctx['name'], 0):,}", 'Flow loss (unitless)')
    register(ctx, '02_train_loss', fig, tcap)

    if ck is not None:
        # 03 checkpoint 对齐的训练 loss（终端指令里的 train_loss_100/1000/5000）
        fig, ax = plt.subplots(figsize=(9, 4.8))
        unique = ck.drop_duplicates('step').sort_values('step')
        for i, width in enumerate(WINDOW_STEPS):
            key = f'train_loss_{width}'
            if unique[key].notna().any():
                ax.plot(unique['step'], unique[key], marker='o', ms=3, lw=1.5,
                        color=('0.65', COLORS['matched'], COLORS['blank'])[i % 3], label=f'trailing {width:,}-step mean')
        finish_axis(ax, 'Checkpoint-aligned training flow loss', 'Flow loss (unitless)')
        register(ctx, '03_checkpoint_train_loss', fig, f"Source: {ctx['name']} loss.log; exact complete windows (step - width, step], rank 0; missing windows are gaps.")

        # 04 各条件动作误差（alt 只覆盖可替换子集）
        metrics = [c for c in ('matched', 'blank', 'wrong', 'alt', 'goal') if c in ck and ck[c].notna().any()]
        fig, axs = plt.subplots(1, len(metrics), figsize=(4.5 * len(metrics), 4.3), squeeze=False)
        for ax, metric in zip(axs[0], metrics):
            evaluation_series(ax, ck, metric)
            finish_axis(ax, f'Held-out {metric} action error', MSE_YLABEL)
        register(ctx, '04_action_errors', fig, eval_caption(ctx) + ' alt covers only eligible queries; do not compare its absolute mean to all-query matched.')

        # 05 三种替换的比值 + CI
        conditions = [c for c in ('blank', 'alt', 'wrong', 'goal') if f'{c}_ratio' in ck and ck[f'{c}_ratio'].notna().any()]
        fig, axs = plt.subplots(1, len(conditions), figsize=(5 * len(conditions), 4.5), squeeze=False)
        for ax, cond in zip(axs[0], conditions):
            ratio_series(ax, ck, cond)
            ratio_refs(ax, cond)
            finish_axis(ax, f'{cond.capitalize()} demo / matched error ratio', RATIO_YLABEL)
        register(ctx, '05_g9_ratios_ci', fig, eval_caption(ctx) + ' CI: query-grouped bootstrap within checkpoint; alt uses its paired eligible subset.')

        # 06 训练 / 留出误差各自归一化的相对变化
        fig, ax = plt.subplots(figsize=(9, 4.8))
        drawn = False
        for i, (queries, group) in enumerate(ck.groupby('queries', sort=True)):
            group = group.sort_values('step')
            tkey = 'train_loss_1000' if group['train_loss_1000'].notna().any() else 'train_loss_100'
            paired = group[np.isfinite(group[tkey]) & np.isfinite(group['matched'])]
            if paired.empty or paired.iloc[0][tkey] <= 0 or paired.iloc[0]['matched'] <= 0:
                continue
            ref = paired.iloc[0]
            for key, lab, color in ((tkey, f'{tkey} / own first', COLORS['blank']), ('matched', 'matched / own first', COLORS['matched'])):
                ax.plot(paired['step'], paired[key] / ref[key], marker=MARKERS[i % 4], lw=1.6, color=color,
                        label=f'{lab}; Q={int(queries)}; ref step {int(ref.step):,}')
            drawn = True
        if drawn:
            ax.axhline(1, color='0.6', lw=0.8, ls=':')
            finish_axis(ax, 'Relative training and held-out error trends', 'Value / own first checkpoint value (unitless)')
            register(ctx, '06_relative_trends', fig, eval_caption(ctx) + ' Each metric normalized separately; descriptive only.')
        else:
            plt.close(fig)

    # 07 辅助训练指标（lr 为第一个参数组；B-4 编码器组 lr 另见配置）
    aux = [('lr', 'Logged learning rate (first param group)', 'Learning rate'), ('grad_norm', 'Gradient norm', 'Gradient norm (unitless)'),
           ('update_s', 'Training update time', 'Update time (s / step)'), ('dataloading_s', 'Data loading time', 'Data loading time (s / step)')]
    aux = [a for a in aux if a[0] in train and train[a[0]].notna().any()]
    if aux:
        fig, axs = plt.subplots(2, 2, figsize=(14, 8), squeeze=False)
        for ax, (metric, title, ylabel) in zip(axs.flat, aux):
            training_series(ax, train, metric, title, ylabel, freq, log=(metric == 'grad_norm'))
            finish_axis(ax, title, ylabel)
        for ax in list(axs.flat)[len(aux):]:
            fig.delaxes(ax)
        register(ctx, '07_training_aux', fig, tcap + ' No GPU-utilization inference.')

    if ck is not None:
        # 08-10 选中 checkpoint 的分组比值
        group_plot(ctx, 'source', 'G9 by data source', '08_source_ratios_ci', ('blank', 'alt', 'wrong'))
        group_plot(ctx, 'robot', 'Blank-demo gain by robot', '09_robot_blank_ci', ('blank',))
        group_plot(ctx, 'pool', 'Alternate-demo ratio by eligible pool size', '10_alt_pool_ci', ('alt',))

        # 11 分来源 blank 比值走势
        g = ctx['groups']
        source = g[(g.dimension == 'source') & (g.condition == 'blank')]
        if len(source) and source['step'].nunique() > 1:
            names = sorted(source['group'].unique())
            ncols = min(3, len(names))
            nrows = math.ceil(len(names) / ncols)
            fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows), squeeze=False)
            for ax, nm in zip(axs.flat, names):
                part = source[source['group'] == nm]
                for i, (queries, group) in enumerate(part.groupby('queries')):
                    group = group.sort_values('step')
                    ax.plot(group.step, group.ratio, marker=MARKERS[i % 4], lw=1.4, ms=3, color=COLORS['blank'],
                            label=f'{int(queries)} total queries')
                    ax.fill_between(group.step.to_numpy(), group.ci_low.to_numpy(dtype=float),
                                    group.ci_high.to_numpy(dtype=float), color=COLORS['blank'], alpha=0.12)
                ax.axhline(1, color='0.5', lw=0.8, ls='--')
                finish_axis(ax, nm, 'Blank / paired matched MSE (unitless)')
            for ax in list(axs.flat)[len(names):]:
                fig.delaxes(ax)
            register(ctx, '11_source_blank_trends', fig, eval_caption(ctx) + ' By-source within-checkpoint 95% CI; group counts in group_ratios.csv.')

    if ck is not None:
        bp_report(ctx)


# ===== BP RoboTwin：分组 / 任务 / 时刻（g9_goal 的 by_group / by_task / by_phase） =====
GROUP_ORDER = ('train', 'train_sibling', 'heldout_sets', 'heldout_tasks')


def bp_report(ctx):
    g = ctx['groups']
    # 12-14 选中 checkpoint 的分组比值 + CI
    group_plot(ctx, 'group', 'G9 by split group (train / held-out candidate sets / held-out tasks)', '12_group_ratios_ci',
               ('blank', 'alt', 'wrong', 'goal'))
    group_plot(ctx, 'task', 'G9 by task', '13_task_ratios_ci', ('blank', 'goal'))
    group_plot(ctx, 'phase', 'G9 by query time (25% / 65% of the episode)', '14_phase_ratios_ci', ('blank', 'goal'))

    # 15 分组走势：matched MSE 与 blank / wrong / goal 比值（每个 query 数各一条线）
    part = g[g.dimension == 'group']
    if part.empty or part.step.nunique() < 2:
        return
    names = [x for x in GROUP_ORDER if x in set(part['group'])] + sorted(set(part['group']) - set(GROUP_ORDER))
    conds = [c for c in ('blank', 'wrong', 'goal') if (part.condition == c).any()]
    fig, axs = plt.subplots(len(names), 1 + len(conds), figsize=(4.6 * (1 + len(conds)), 3.4 * len(names)), squeeze=False)
    for r, nm in enumerate(names):
        sel = part[part['group'] == nm]
        ax = axs[r, 0]
        for i, (queries, grp) in enumerate(sel[sel.condition == 'matched'].groupby('queries')):
            grp = grp.sort_values('step')
            ax.plot(grp.step, grp.mse, marker=MARKERS[i % 4], linestyle=LINESTYLES[i % 4], lw=1.6, ms=4,
                    color=COLORS['matched'], label=f'matched; {int(queries)} total queries')
        finish_axis(ax, f'{nm}: matched action error', MSE_YLABEL)
        for ax, cond in zip(axs[r, 1:], conds):
            rows_c = sel[sel.condition == cond]
            if rows_c.empty:
                ax.axis('off')
                ax.text(0.5, 0.5, f'{nm}: no {cond} condition\n(no contrast set / not in this query set)',
                        ha='center', va='center', color='0.5', transform=ax.transAxes)
                continue
            for i, (queries, grp) in enumerate(rows_c.groupby('queries')):
                grp = grp.sort_values('step')
                ax.plot(grp.step, grp.ratio, marker=MARKERS[i % 4], linestyle=LINESTYLES[i % 4], lw=1.6, ms=4,
                        color=COLORS[cond], label=f'{cond}; {int(queries)} total queries')
                ax.fill_between(grp.step.to_numpy(), grp.ci_low.to_numpy(dtype=float), grp.ci_high.to_numpy(dtype=float),
                                color=COLORS[cond], alpha=0.12)
            ratio_refs(ax, cond)
            finish_axis(ax, f'{nm}: {cond} / matched', RATIO_YLABEL)
    register(ctx, '15_group_trends', fig, eval_caption(ctx) + ' Per split group; groups differ between the two G9 query sets '
             '(train_sibling only in 600-query g9_main; train and goal only in 444-query g9_goal).')


for ctx in CTX.values():
    print(f"\n######## {ctx['title']} ########")
    stage_report(ctx)

## 跨阶段对比（B-4 SFT → B-4 SFT full）

横轴为累计步数：各阶段步数 + `start_step`（B-4 的第 0 步 = B-3 的 105k）。竖点线标出各阶段起点。

- 训练 loss：各阶段的 1,000 步 trailing mean。B-4 为新优化器（warmup 500），可观察切换处是否有跳变。
- G9：matched 及 blank / alt / wrong 比值（含 CI）。B-4 与 B-3 的 G9 可比（见上方指标口径）。只有各阶段用相同 query 数时才适合直接比较曲线高低。
- 汇总表：各阶段首 / 末 / 最低 matched 及末次比值，只是描述，不作显著性判定。

In [ ]:
COMPARE = {'title': 'Stage B: ' + ' -> '.join(CTX), 'figures': {}, 'notes': [], 'report_dir': COMPARE_DIR}
CUM_XLABEL = 'Cumulative step from B-4 SFT start (k)'
TRAINED = list(CTX.values())
EVALUATED = [c for c in CTX.values() if c['ckpt'] is not None]
SUMMARY = None


def stage_color(ctx):
    return STAGE_COLORS.get(ctx['name'], f'C{list(CTX).index(ctx["name"])}')


def stage_starts(ax):
    for ctx in CTX.values():
        if ctx['start']:
            ax.axvline(ctx['start'], color=stage_color(ctx), ls=':', lw=1.0, label=f"{ctx['name']} start = step {ctx['start']:,}")


def compare_caption(stages):
    parts = [f"{c['name']} {c['run'].name} (offset {c['start']:,})" for c in stages]
    return 'Source: ' + '; '.join(parts) + '. x = stage step + offset.'


def until_next_stage(ctx, df):
    # 一个阶段在下一阶段的起点截止（如 B-3 中断前的 105k-107.4k 不是 B-4 的前序）
    later = [c['start'] for c in CTX.values() if c['start'] > ctx['start']]
    return df[df['step'] + ctx['start'] <= min(later)] if later else df


def stage_bridges(ax, metric):
    # 用虚线连接阶段切换两端的实测点，不补造第 0 步评估，也不跨阶段填充 CI。
    stages = sorted(CTX.values(), key=lambda c: c['start'])
    for previous, current in zip(stages, stages[1:]):
        if previous['ckpt'] is None or current['ckpt'] is None:
            continue
        left = until_next_stage(previous, previous['ckpt'])
        right = until_next_stage(current, current['ckpt'])
        if metric not in left or metric not in right:
            continue
        # 前一阶段必须有恰好位于初始化 checkpoint 的评估；query 数也必须一致。
        boundary = left[left['step'] + previous['start'] == current['start']]
        for queries, group in right.groupby('queries', sort=True):
            anchor = boundary[boundary['queries'] == queries]
            if anchor.empty:
                continue
            first = group.sort_values('step').iloc[0]
            x = [current['start'], int(first['step']) + current['start']]
            y = [anchor.iloc[-1][metric], first[metric]]
            if x[1] <= x[0] or not np.isfinite(y).all():
                continue
            ax.plot(x, y, color=stage_color(current), ls='--', lw=1.6,
                    label=f"{previous['name']} -> {current['name']}; {int(queries)} queries")


if len(TRAINED) >= 2:
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for ctx in TRAINED:
        df = until_next_stage(ctx, ctx['train'])
        ax.plot(df['step'] + ctx['start'], df['loss'], color=stage_color(ctx), lw=0.7, alpha=0.25,
                label=f"{ctx['name']} logged {ctx['freq']}-step mean")
        ax.plot(df['step'] + ctx['start'], train_smooth(df, 'loss', ctx['freq']), color=stage_color(ctx), lw=1.8,
                label=f"{ctx['name']} trailing {SMOOTH_STEPS:,}-step mean")
    stage_starts(ax)
    finish_axis(ax, 'Training flow-matching loss across stages', 'Flow loss (unitless)', CUM_XLABEL)
    register(COMPARE, 'C1_train_loss', fig, compare_caption(TRAINED) + ' Rank-0 loss.log; B-4 SFT full restarts the optimizer (warmup 500) from the LoRA-merged B-4 SFT 20k.')

if len(EVALUATED) >= 2:
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for ctx in EVALUATED:
        evaluation_series(ax, until_next_stage(ctx, ctx['ckpt']), 'matched', color=stage_color(ctx), offset=ctx['start'],
                          label=f"{ctx['name']} ")
    stage_bridges(ax, 'matched')
    stage_starts(ax)
    finish_axis(ax, 'Held-out matched action error across stages', MSE_YLABEL, CUM_XLABEL)
    register(COMPARE, 'C2_matched', fig, compare_caption(EVALUATED) + ' G9 matched MSE; dashed bridges join stage endpoints at equal query counts (not verified cohort identity).')

    conditions = [c for c in ('blank', 'alt', 'wrong', 'goal') if any(f'{c}_ratio' in x['ckpt'] for x in EVALUATED)]
    fig, axs = plt.subplots(1, len(conditions), figsize=(5.5 * len(conditions), 4.6), squeeze=False)
    for ax, cond in zip(axs[0], conditions):
        for i, ctx in enumerate(EVALUATED):
            ratio_series(ax, until_next_stage(ctx, ctx['ckpt']), cond, color=stage_color(ctx), offset=ctx['start'],
                         label=f"{ctx['name']} ", ci_label=(i == 0))
        stage_bridges(ax, f'{cond}_ratio')
        ratio_refs(ax, cond)
        stage_starts(ax)
        finish_axis(ax, f'{cond.capitalize()} demo / matched error ratio', RATIO_YLABEL, CUM_XLABEL)
    register(COMPARE, 'C3_g9_ratios', fig, compare_caption(EVALUATED) + ' Shaded: within-checkpoint 95% CI; dashed bridges join stage endpoints only (no cross-stage CI).')

    rows = []
    for ctx in EVALUATED:
        for queries, g in ctx['ckpt'].groupby('queries'):
            g = g.sort_values('step')
            first, last, best = g.iloc[0], g.iloc[-1], g.loc[g['matched'].idxmin()]
            row = {'stage': ctx['name'], 'queries': int(queries), 'evaluations': len(g),
                   'first_step': int(first.step), 'last_step': int(last.step),
                   'matched_first': first.matched, 'matched_last': last.matched,
                   'matched_min': best.matched, 'matched_min_step': int(best.step),
                   'train_loss_1000_last': last.get('train_loss_1000', np.nan)}
            for cond in ('blank', 'alt', 'wrong', 'goal'):
                if f'{cond}_ratio' in g:
                    row[f'{cond}_ratio_last'] = last[f'{cond}_ratio']
                    row[f'{cond}_ci_last'] = f"[{last[f'{cond}_ci_low']:.3f}, {last[f'{cond}_ci_high']:.3f}]"
            rows.append(row)
    SUMMARY = pd.DataFrame(rows)
    print('各阶段汇总（描述性，按阶段自己的步数）：')
    display(SUMMARY.round(4))

if not COMPARE['figures']:
    print('对比图需要至少两个阶段有训练日志（或 G9）；当前有数据的阶段：' + ', '.join(CTX))

## 导出

- 各阶段：`<run>/reports/checkpoint_eval/` 下 `01_overview` … `11_source_blank_trends` 的 PNG / PDF / SVG（没有数据的图省略），`training_metrics.csv`、`checkpoint_metrics.csv`、`group_ratios.csv`（后两个需要有 G9）和 `manifest.json`。
- 对比：`stage_b/reports/stage_compare/` 下 `C1_train_loss`、`C2_matched`、`C3_g9_ratios`，`stage_summary.csv` 和 `manifest.json`。
- `manifest.json` 记录输入路径、窗口、版本、源文件大小 / 修改时间和每张图的说明；训练仍在进行时，以导出的 CSV 为本次报告的快照。

In [ ]:
NOTES = [
    'Training loss is rank-0 interval-averaged flow loss, not an 8-GPU global mean.',
    'G9 errors are normalized-delta action MSE, not physical EE distances.',
    'Ratios / CI are read from paired-subset query-bootstrap results, never recomputed from all-query means.',
    'BP G9: demos are P64 ViT records pooled to the P5 tokens of the demo encoder (online encoding).',
    'goal = demo of the same anchor with another goal (contrast candidate set); only tasks with >= 2 goals.',
    'Two G9 query sets (600 g9_main, 444 g9_goal) are drawn separately; groups / conditions differ between them.',
    'Query IDs and seeds are not stored in the JSON; equal query counts do not verify cohort identity.',
    'No matched-error CI or cross-checkpoint significance is inferred.',
    'Source files may still be changing; the exported CSV files define the plotted snapshot.',
]


def source_info(paths):
    out = []
    for path in paths:
        stat = Path(path).stat()
        out.append({'path': str(path), 'bytes_at_export': stat.st_size,
                    'mtime_utc_at_export': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat()})
    return out


def export(target, tables, manifest):
    d = Path(target['report_dir'])
    d.mkdir(parents=True, exist_ok=True)
    for name, fig in target['figures'].items():
        for ext in FORMATS:
            fig.savefig(d / f'{name}.{ext}', dpi=DPI, bbox_inches='tight')
    for fname, df in tables.items():
        df.to_csv(d / fname, index=False)
    manifest.update({'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'formats': list(FORMATS), 'dpi': DPI,
                     'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__},
                     'notes': NOTES, 'figures': target['notes']})
    (d / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    print(f"已导出 {len(target['figures'])} 组图 × {len(FORMATS)} 格式 + {len(tables)} 份 CSV + manifest.json -> {d}")


if EXPORT:
    if not set(FORMATS).issubset({'png', 'pdf', 'svg'}):
        raise ValueError('FORMATS 仅支持 png / pdf / svg')
    common = {'g9_dir': str(G9_DIR), 'exclude_files': sorted(EXCLUDE_FILES), 'query_counts': QUERY_COUNTS,
              'min_step': MIN_STEP, 'max_step': MAX_STEP, 'smooth_steps': SMOOTH_STEPS, 'train_window_steps': list(WINDOW_STEPS)}
    for ctx in CTX.values():
        tables = {'training_metrics.csv': ctx['train']}
        if ctx['ckpt'] is not None:
            tables.update({'checkpoint_metrics.csv': ctx['ckpt'], 'group_ratios.csv': ctx['groups']})
        paths = [ctx['run'] / 'loss.log'] + [r['path'] for r in ctx['records']] + ([ctx['cfg_file']] if ctx['cfg_file'] else [])
        export(ctx, tables, {**common, 'stage': ctx['name'], 'run': str(ctx['run']), 'log_freq': ctx['freq'],
                             'start_step': ctx['start'], 'selected_checkpoint': ctx['select'],
                             'read_warnings': ctx['issues'], 'sources': source_info(paths)})
    if COMPARE['figures']:
        export(COMPARE, {'stage_summary.csv': SUMMARY} if SUMMARY is not None else {},
               {**common, 'stages': [{'name': c['name'], 'run': str(c['run']), 'start_step': c['start'],
                                      'evaluations': len(c['records'])} for c in CTX.values()]})
else:
    print('EXPORT=False：只显示图，不写文件。')

## 附：BP 逐任务与仿真测评对照

以下为逐任务 G9（`g9_goal` 最后一个 checkpoint 的逐条明细）、仿真测评与两者对照；需要先运行第一格（输入）。


In [ ]:
from IPython.display import Markdown
G9_SOURCES = {'g9_main': 'results/step*.json', 'g9_goal': '2026-10-29_18-26_*k.json'}
SIM_EVALS = [Path('/vla/workspace/my_robotwin/eval_result_bp/b4_sft_v1_20k')]   # 仿真测评目录，可多个
COND_COLORS = {'blank': COLORS['blank'], 'alt': COLORS['alt'], 'wrong': COLORS['wrong'], 'goal': COLORS['goal']}


## 3 逐任务（`g9_goal` 最后一个 checkpoint）

In [ ]:
goal_files = sorted(G9_DIR.glob(G9_SOURCES['g9_goal']), key=lambda p: int(json.loads(p.read_text())['ckpt'].split('/')[-2]))
goal_files = [p for p in goal_files if not p.name.endswith('_rows.jsonl')]
TASKS = pd.DataFrame()
if goal_files:
    last_file = goal_files[-1]
    rows = pd.read_json(last_file.with_name(last_file.stem + '_rows.jsonl'), lines=True)
    print('file:', last_file.name, '| samples', len(rows))
    agg = {'matched': 'mean', 'blank': 'mean', 'wrong': 'mean', 'alt': 'mean', 'goal': 'mean', 'q': 'nunique'}
    TASKS = rows.groupby(['task', 'group']).agg({k: v for k, v in agg.items() if k in rows}).rename(columns={'q': 'queries'})
    for c in ('blank', 'wrong', 'alt', 'goal'):
        if c in TASKS:
            TASKS[f'{c}/matched'] = TASKS[c] / TASKS['matched']
    TASKS = TASKS.reset_index().sort_values(['group', 'goal/matched' if 'goal/matched' in TASKS else 'matched'])
    display(TASKS.round(3))

    # phase: does the demo matter early (before the goal is visible in the scene) or late?
    phase = rows.groupby(['group', 'phase'])[['matched', 'blank', 'goal']].mean()
    phase['blank/matched'] = phase.blank / phase.matched
    phase['goal/matched'] = phase.goal / phase.matched
    display(phase.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(17, 0.28 * len(TASKS) + 1.5))
    t = TASKS.sort_values('matched')
    lab = t.task.str.replace('_bp', '') + ' [' + t.group.str.replace('heldout_', 'ho_') + ']'
    axes[0].barh(lab, t.matched, color=t.group.map({'train': 'C0', 'heldout_sets': 'C1', 'heldout_tasks': 'C3'}))
    axes[0].set_title('matched MSE per task (colour = group)')
    for c in ('blank/matched', 'goal/matched'):
        if c in t:
            axes[1].scatter(t[c], lab, label=c, color=COND_COLORS[c.split('/')[0]])
    axes[1].axvline(1.0, color='k', lw=0.8, ls='--'); axes[1].set_xscale('log'); axes[1].legend(frameon=False)
    axes[1].set_title('ratio per task (> 1: matched demo helps)')
    plt.tight_layout(); plt.show()

## 4 仿真测评

In [ ]:
SIM = []
for d in SIM_EVALS:
    p = d / '_summary.json'
    if not p.is_file():
        print('missing', p); continue
    s = json.loads(p.read_text())
    meta = json.loads((d / 'server_metadata.json').read_text()) if (d / 'server_metadata.json').is_file() else {}
    print(f"{d.name}: {s['valid']} valid episodes, env errors {s['env_errors']} | ckpt {meta.get('checkpoint_dir', '?')}")
    display(pd.DataFrame(s['groups']).T.assign(rate=lambda x: x.success / x.n).round(3))
    for key, v in s['tasks'].items():
        task, group = key.split('|')
        c = s.get('contrast', {}).get(key, {})
        SIM.append({'eval': d.name, 'task': task, 'group': group, 'success': v['success'], 'n': v['n'],
                    'rate': v['success'] / v['n'], 'contrast_solved': c.get('solved'), 'contrast_n': c.get('n')})
SIM = pd.DataFrame(SIM)
if not SIM.empty:
    display(SIM.sort_values(['group', 'rate'], ascending=[True, False]).round(3))
    eps = pd.concat([pd.read_json(p, lines=True).assign(eval=d.name) for d in SIM_EVALS
                     for p in sorted(d.glob('*/episodes.jsonl'))], ignore_index=True)
    eps['timeout'] = ~eps.success & (eps.steps >= eps.step_lim)
    print('failed episodes that ran to the step limit:', int(eps.timeout.sum()), '/', int((~eps.success).sum()))
    display(eps.groupby(['group', 'goal']).success.agg(['mean', 'size']).sort_values('mean').head(30).round(3))

## 5 离线 G9 vs 仿真成功率（逐任务）

In [ ]:
if not TASKS.empty and not SIM.empty:
    j = SIM.merge(TASKS, on=['task', 'group'], how='left')
    cols = ['eval', 'task', 'group', 'rate', 'n', 'contrast_solved', 'contrast_n', 'matched', 'blank/matched', 'goal/matched']
    display(j[[c for c in cols if c in j]].sort_values(['group', 'rate']).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, x in zip(axes, ('matched', 'goal/matched')):
        if x in j:
            for g, d in j.groupby('group'):
                ax.scatter(d[x], d.rate, label=g, s=18 + 4 * d.n)
            ax.set_xlabel(x + ' (G9, last ckpt)'); ax.set_ylabel('sim success rate'); ax.legend(frameon=False)
    axes[1].axvline(1.0, color='k', lw=0.8, ls='--'); axes[1].set_xscale('log')
    plt.tight_layout(); plt.show()
else:
    print('need both g9_goal results and a sim eval')

## 附：快速仿真测评随累计步数

同样 6 个任务、各 5 个训练场景，每个 checkpoint 30 回合；两轮的 checkpoint 接在同一条累计步数轴上（第二轮 +20k）。


In [ ]:
# ===== 快速仿真测评随累计步数（tools/bpv4cdxb/sim_sweep_bp.sh：6 个任务 x 5 个训练场景） =====
SWEEPS = {'b4sft_sft_': 0, 'b4full_full_': 20000}   # 测评目录前缀 -> 累计步数偏移
SIM_ROOT = Path('/vla/workspace/my_robotwin/eval_result_bp')
rows = []
for prefix, offset in SWEEPS.items():
    for d in sorted(SIM_ROOT.glob(prefix + '*k')):
        step = int(d.name[len(prefix):-1]) * 1000
        for p in d.glob('*/episodes.jsonl'):
            for line in p.read_text().splitlines():
                if line.strip():
                    r = json.loads(line)
                    rows.append({'run': prefix.rstrip('_'), 'step': step, 'cum_step': step + offset,
                                 'task': r['task'].replace('_bp', ''), 'success': bool(r['success'])})
SWEEP = pd.DataFrame(rows)
if SWEEP.empty:
    print('no sweep results yet')
else:
    display(SWEEP.pivot_table(index='task', columns='cum_step', values='success', aggfunc='mean').round(2))
    tot = SWEEP.groupby(['run', 'cum_step']).success.agg(['mean', 'sum', 'size']).reset_index()
    display(tot)
    fig, axs = plt.subplots(1, 2, figsize=FIGSIZE_ROW)
    run_color = {'b4sft_sft': STAGE_COLORS['B-4 SFT'], 'b4full_full': STAGE_COLORS['B-4 SFT full']}
    for run, g in tot.groupby('run'):
        axs[0].plot(g.cum_step, g['mean'], marker='o', color=run_color.get(run), label=f'{run} (n={int(g["size"].iloc[0])} per ckpt)')
    axs[0].axvline(20000, color=STAGE_COLORS['B-4 SFT full'], ls=':', lw=1.0, label='second run start')
    finish_axis(axs[0], 'Sim success on training scenes (6 tasks)', 'Success rate', 'Cumulative post-training step (k)')
    for task, g in SWEEP.groupby('task'):
        g = g.groupby('cum_step').success.mean()
        axs[1].plot(g.index, g.values, marker='o', ms=3, lw=1.2, label=task)
    axs[1].axvline(20000, color='0.5', ls=':', lw=1.0)
    finish_axis(axs[1], 'Per task', 'Success rate', 'Cumulative post-training step (k)')
    plt.tight_layout(); plt.show()


## 怎么读（排查思路）

| 现象 | 可能的问题 |
| --- | --- |
| 训练 loss 还在明显下降 | 欠训练：步数不够（本 run 只走了约 0.6 个 epoch） |
| matched MSE 低、blank / goal 比值明显 > 1，但仿真成功率低 | 离线逐步预测没问题，**闭环**出问题：误差累积、动作执行步数（客户端每次执行 16 / 24 / 30 步，训练 n_action_steps = 25）、测评场景与训练场景的分布差（偏移场景、jitter ≥ 10000） |
| blank 比值 > 1 但 **goal 比值 ≈ 1** | 读了示教，但没读出**目标**：同锚点不同目标时动作差不多 → 对比组无法全对，只能靠运气做对一半 |
| 留出任务 blank 比值 < 1 | 没见过的任务上示教在误导（示教编码器 / 读取方式没有泛化），属于零样本能力缺失 |
| 25% 时刻的 goal 比值明显小于 65% | 早期动作和目标无关（两目标的早期动作本来就相似），这本身正常；若 65% 也 ≈ 1 才是真问题 |
| 个别任务 matched MSE 远高于同组 | 该任务本身难拟合（长程、双臂、精细），优先看它的视频 |